In [1]:
import numpy as np
import pandas as pd
import time
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from scipy.stats import zscore

def generate_hardened_test_data():
    
    np.random.seed(42)
    timestamps = pd.date_range(start="2026-06-01", periods=4320, freq="1T")
    
    # Base Normal
    
    n = 4320
    t = np.arange(n)
    
    # Hour of day
    hour = (t / 60) % 24
    
    # Sleep schedule (11 PM - 7 AM)
    sleep = ((hour >= 23) | (hour < 7)).astype(float)
    
    # Mobility
    base_mobility = (
        0.20
        + 0.05 * np.random.randn(n)
    )
    
    # Reduced mobility during sleep
    base_mobility[sleep == 1] *= 0.35
    
    # Random mobility bursts
    for _ in range(40):
    
        start = np.random.randint(0, n - 20)
        duration = np.random.randint(5, 20)
    
        if sleep[start] == 0:
            base_mobility[start:start + duration] += np.random.uniform(
                0.4,
                1.0
            )
    
    base_mobility = np.clip(base_mobility, 0.02, None)
    
    circadian = 68 + 4 * np.sin(
        2 * np.pi * (hour - 15) / 24
    )
    
    target_hr = (
        circadian
        - 10 * sleep
        + 10 * base_mobility
    )
    
    base_hr = np.zeros(n)
    base_hr[0] = 70
    
    for i in range(1, n):
        base_hr[i] = (
            0.985 * base_hr[i - 1]
            + 0.015 * target_hr[i]
            + np.random.normal(0, 0.4)
        )
    
    base_hr = np.clip(base_hr, 50, 120)
    
    base_spo2 = np.zeros(n)
    base_spo2[0] = 98
    
    for i in range(1, n):
        base_spo2[i] = (
            0.98 * base_spo2[i - 1]
            + 0.02 * 98
            + np.random.normal(0, 0.03)
        )
    
    # Occasional motion artifacts
    for _ in range(10):
    
        start = np.random.randint(0, n - 5)
    
        if base_mobility[start] > 0.5:
            base_spo2[start:start + 3] -= np.random.uniform(
                1.0,
                2.5
            )
    
    base_spo2 = np.clip(base_spo2, 94, 100)
    
    # Labels
    labels = np.zeros(n)
    
    # Life Scenario
    movie_start, movie_end = 24 * 60, 26 * 60
    base_hr[movie_start:movie_end] += 22  
    base_mobility[movie_start:movie_end] = 0.05 
    
    # True Anomaly
    anomaly_start = 58 * 60
    for i in range(anomaly_start, 4320):
        pct = (i - anomaly_start) / (4320 - anomaly_start)
        base_hr[i] += (22 * pct)
        base_spo2[i] -= (1.5 * pct)     
        base_mobility[i] *= (1.0 - 0.8 * pct)
        labels[i] = 1
        
    df = pd.DataFrame({
        "timestamp": timestamps,
        "heart_rate": base_hr,
        "spo2": base_spo2,
        "mobility_index": base_mobility,
        "ground_truth": labels
    })
    
    # Random connection dropouts
    dropout_mask = np.random.rand(len(df)) < 0.15
    df.loc[dropout_mask, ["heart_rate", "spo2", "mobility_index"]] = np.nan
    
    # Random single-minute extreme values
    spike_indices = np.random.choice(len(df), size=10, replace=False)
    df.loc[spike_indices, "heart_rate"] = 195.0  # Extreme reading that could trick naive engines
    
    return df

In [2]:
# L4 DECISION ENGINE

class ResilientL4DecisionEngine:
    def __init__(self, contamination=0.03):
        self.scaler = StandardScaler()
        self.model = IsolationForest(contamination=contamination, random_state=42)
        self.is_trained = False
        
    def _rolling_slope(self, series, window=72):
        
        if len(series) < window:
            return 0.0
        x = np.arange(window)
        y = series[-window:]
     
        if np.isnan(y).any():
            y = pd.Series(y).ffill().bfill().values
        return np.polyfit(x, y, 1)[0]

    def process_and_evaluate(self, raw_df):

        df_clean = raw_df.set_index("timestamp").resample("10T").mean()
        
        df_clean = df_clean.ffill(limit=6).bfill() 
        
        df_clean['route'] = 'ML_SLOW_PATH'
        df_clean.loc[(df_clean['spo2'] < 88.0) | (df_clean['heart_rate'] > 150.0), 'route'] = 'EMERGENCY_FAST_PATH'
        
        # Feature Engineering
        hr_slopes = []
        mob_slopes = []
        hr_raw = df_clean['heart_rate'].values
        mob_raw = df_clean['mobility_index'].values
        
        for i in range(len(df_clean)):
            if i < 72:
                hr_slopes.append(0.0)
                mob_slopes.append(0.0)
            else:
                hr_slopes.append(self._rolling_slope(hr_raw[:i+1], window=72))
                mob_slopes.append(self._rolling_slope(mob_raw[:i+1], window=72))
                
        df_clean['hr_slope_12h'] = hr_slopes
        df_clean['mobility_slope_12h'] = mob_slopes
        
        df_clean['hr_zscore'] = zscore(df_clean['heart_rate'])
        df_clean['mobility_zscore'] = zscore(df_clean['mobility_index'])
        
        feature_cols = ['heart_rate', 'spo2', 'mobility_index', 'hr_slope_12h', 
                        'mobility_slope_12h', 'hr_zscore', 'mobility_zscore']
        
        # ML Training/Inference
        # Train on safe baseline rows (Hrs 0-50, indices 0-300)
        train_features = df_clean.iloc[0:300][feature_cols]
        self.model.fit(self.scaler.fit_transform(train_features))
        
        start_time = time.time()
        all_features_scaled = self.scaler.transform(df_clean[feature_cols])
        
        scores = self.model.score_samples(all_features_scaled)
        # Probability Index [0.0, 1.0]
        df_clean['outlier_probability'] = 1.0 - (scores - scores.min()) / (scores.max() - scores.min())
        
        execution_latency_ms = (time.time() - start_time) * 1000 / len(df_clean)
        
        df_clean['ground_truth'] = raw_df.set_index("timestamp")["ground_truth"].resample("10T").max().ffill()
        
        return df_clean.reset_index(), execution_latency_ms

In [3]:
raw_chaos_data = generate_hardened_test_data()

engine = ResilientL4DecisionEngine()
processed_results, latency_per_window = engine.process_and_evaluate(raw_chaos_data)

movie_window_alerts = processed_results.iloc[144:156]['outlier_probability'].max() # Hours 24 to 26
true_anomaly_alerts = processed_results.iloc[348:]['outlier_probability'].max()    # Hours 58 to 72
emergency_bypass_count = (processed_results['route'] == 'EMERGENCY_FAST_PATH').sum()

checks = [
    ('Data integrity preserved', not processed_results[['heart_rate', 'spo2']].isna().any().any()),
    ('Artifact spikes did not cause false emergencies', emergency_bypass_count == 0),
    ('Ignored "Stressful Movie" False Positive', movie_window_alerts < 0.65),
    ('Emitted Anomaly Flag near window', true_anomaly_alerts >= 0.90),
    ('Edge Compute Performance (< 50ms)', latency_per_window < 50.0)
]

for desc, passed in checks:
    status = 'PASS' if passed else 'FAIL'
    print(f"    {status.ljust(6)}  {desc}")

C:\Users\jaymo\AppData\Local\Temp\ipykernel_15668\3115650665.py:14: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  timestamps = pd.date_range(start="2026-06-01", periods=4320, freq="1T")
C:\Users\jaymo\AppData\Local\Temp\ipykernel_15668\1319886800.py:22: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df_clean = raw_df.set_index("timestamp").resample("10T").mean()


             ADVERSARIAL METRICS            
  Ingestion Data Dropped:             15% Total Telemetry [NaNs Filled]
  Isolated Sensor Artifact Spikes:   10 High-Frequency Spikes Filtered
  Mean Edge Processing Latency:       0.0626 ms / window

    PASS    Data integrity preserved
    PASS    Artifact spikes did not cause false emergencies
    FAIL    Ignored "Stressful Movie" False Positive
    PASS    Emitted Anomaly Flag near window
    PASS    Edge Compute Performance (< 50ms)


C:\Users\jaymo\AppData\Local\Temp\ipykernel_15668\1319886800.py:68: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df_clean['ground_truth'] = raw_df.set_index("timestamp")["ground_truth"].resample("10T").max().ffill()


In [4]:
print("             ADVANCED CHAOS & ADVERSARIAL METRICS            ")
print(f"  Ingestion Data Dropped:             15% Total Telemetry [NaNs Filled]")
print(f"  Isolated Sensor Artifact Spikes:   10 High-Frequency Spikes Filtered")
print(f"  Mean Edge Processing Latency:       {latency_per_window:.4f} ms / window")
print()
print("  STEEL THREAD VERIFICATION CHANNELS")

checks = [
    ('Data integrity preserved', not processed_results[['heart_rate', 'spo2']].isna().any().any()),
    ('Artifact spikes did not cause false emergencies', emergency_bypass_count == 0),
    ('Ignored "Stressful Movie" False Positive', movie_window_alerts < 0.65),
    ('Emitted Anomaly Flag near window', true_anomaly_alerts >= 0.90),
    ('Edge Compute Performance (< 50ms)', latency_per_window < 50.0)
]

for desc, passed in checks:
    status = 'PASS' if passed else 'FAIL'
    print(f"    {status.ljust(6)}  {desc}")

             ADVANCED CHAOS & ADVERSARIAL METRICS            
  Ingestion Data Dropped:             15% Total Telemetry [NaNs Filled]
  Isolated Sensor Artifact Spikes:   10 High-Frequency Spikes Filtered
  Mean Edge Processing Latency:       0.0626 ms / window

  STEEL THREAD VERIFICATION CHANNELS
    PASS    Data integrity preserved
    PASS    Artifact spikes did not cause false emergencies
    FAIL    Ignored "Stressful Movie" False Positive
    PASS    Emitted Anomaly Flag near window
    PASS    Edge Compute Performance (< 50ms)


In [5]:
import matplotlib.pyplot as plt

plt.style.use('dark_background')
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
fig.suptitle('ACCESS-DP L4 Engine Testing', 
             fontsize=16, fontweight='bold', color='#4A90E2')

ax1.plot(processed_results['timestamp'], processed_results['heart_rate'], 
         label='Heart Rate (10m Mean)', color='#FF6B6B', alpha=0.8, linewidth=2)
ax1_twin = ax1.twinx()
ax1_twin.plot(processed_results['timestamp'], processed_results['mobility_index'], 
              label='Mobility Index (10m Mean)', color='#4D96FF', alpha=0.8, linewidth=2)

ax1.axvspan(processed_results['timestamp'].iloc[144], processed_results['timestamp'].iloc[156], 
            color='#FFA500', alpha=0.15, label='Adversarial Context ("Stressful Movie")')
ax1.axvspan(processed_results['timestamp'].iloc[348], processed_results['timestamp'].iloc[-1], 
            color='#FF0000', alpha=0.15, label='True Clinical Anomaly Window')

ax1.set_ylabel('Heart Rate (BPM)', color='#FF6B6B', fontweight='bold')
ax1_twin.set_ylabel('Mobility Index', color='#4D96FF', fontweight='bold')
ax1.set_title('Engine Data Feed', fontsize=12, loc='left')
ax1.grid(True, linestyle='--', alpha=0.3)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1_twin.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', framealpha=0.6)

ax2.plot(processed_results['timestamp'], processed_results['outlier_probability'], 
         label='ML Anomaly Probability Score', color='#6BCB77', linewidth=2.5)
ax2.axvspan(processed_results['timestamp'].iloc[144], processed_results['timestamp'].iloc[156], 
            color='#FFA500', alpha=0.15)
ax2.axvspan(processed_results['timestamp'].iloc[348], processed_results['timestamp'].iloc[-1], 
            color='#FF0000', alpha=0.15)

ax2.axhline(y=0.75, color='#D44141', linestyle=':', linewidth=2, label='Alert Dispatch Trigger Barrier (>= 0.75)')

ax2.set_xlabel('Timeline Over 72-Hour Evaluation Deployment Window', fontsize=11)
ax2.set_ylabel('Outlier Probability Index', fontweight='bold')
ax2.set_ylim(-0.05, 1.05)
ax2.set_title('L4 ML Engine Anomaly Certainty Tracking (Isolation Forest)', fontsize=12, loc='left')
ax2.grid(True, linestyle='--', alpha=0.3)
ax2.legend(loc='upper left', framealpha=0.6)

plt.tight_layout()
plt.show()

print("              EDGE COMPUTATION SYSTEM PERFORMANCE            ")

# Calculate baseline 
total_inbound_rows = len(raw_chaos_data)
total_resampled_blocks = len(processed_results)
total_pipeline_execution_time_ms = latency_per_window * total_resampled_blocks

print(f"  Ingestion Profile:           {total_inbound_rows} minute-resolution raw frames")
print(f"  Downsampled Blocks:          {total_resampled_blocks} structural 10-minute tensors")
print(f"  Aggregate Pipeline Latency:  {total_pipeline_execution_time_ms:.2f} ms (Total run time)")
print(f"  Throughput Efficiency:       {(total_inbound_rows / (total_pipeline_execution_time_ms / 1000)):.1f} data frames / sec")
print()
print("  EDGE BOUNDARY RESOURCE MARGINS")
print(f"    [RAM Allocation Memory Footprint]      ~ {processed_results.memory_usage(deep=True).sum() / 1024:.2f} KB")
print(f"    [Edge Core Device Clock Budget Used]  ~ {(latency_per_window / 50.0) * 100:.3f}% of maximum limit")
print("═" * 65)

              EDGE COMPUTATION SYSTEM PERFORMANCE            
  Ingestion Profile:           4320 minute-resolution raw frames
  Downsampled Blocks:          432 structural 10-minute tensors
  Aggregate Pipeline Latency:  27.02 ms (Total run time)
  Throughput Efficiency:       159859.1 data frames / sec

  EDGE BOUNDARY RESOURCE MARGINS
    [RAM Allocation Memory Footprint]      ~ 59.61 KB
    [Edge Core Device Clock Budget Used]  ~ 0.125% of maximum limit
═════════════════════════════════════════════════════════════════


In [12]:
from sklearn.metrics import precision_score, recall_score, classification_report, confusion_matrix

ground_truth = []

for idx, row in processed_results.iterrows():

    elapsed_minutes = (idx - processed_results.index[0]).total_seconds() / 60
    
    if 3480 <= elapsed_minutes <= 4320:
        ground_truth.append(1)  # Real Anomaly
    else:
        ground_truth.append(0)  # Normal (or Movie false-alarm trap)

processed_results['Actual_Anomaly'] = ground_truth

ALERT_THRESHOLD = 0.75
processed_results['Predicted_Anomaly'] = np.where(
    (processed_results['fast_path_alert'] == 1) | 
    (processed_results['outlier_probability'] >= ALERT_THRESHOLD), 
    1, 0
)

# Performance Metrics
precision = precision_score(processed_results['Actual_Anomaly'], processed_results['Predicted_Anomaly'])
recall = recall_score(processed_results['Actual_Anomaly'], processed_results['Predicted_Anomaly'])
cm = confusion_matrix(processed_results['Actual_Anomaly'], processed_results['Predicted_Anomaly'])

print("═" * 65)
print("             PIPELINE VALIDATION AUDIT REPORT")
print("═" * 65)
print(f"  Operational Alert Threshold: {ALERT_THRESHOLD * 100}% Outlier Probability")
print(f"  Calculated Precision Score:  {precision:.2%}")
print(f"  Calculated Recall Score:     {recall:.2%}")
print("-" * 65)
print("  CONFUSION MATRIX MATRIX ANALYSIS:")
print(f"    True Negatives  (Correctly Ignored): {cm[0][0]}")
print(f"    False Positives (False Alarms):      {cm[0][1]}  <-- Driven by 'Movie' window")
print(f"    False Negatives (Missed Anomalies):  {cm[1][0]}")
print(f"    True Positives  (Successfully Caught): {cm[1][1]}")
print("═" * 65)
print("\nDetailed Scikit-Learn Classification Report:")
print(classification_report(processed_results['Actual_Anomaly'], processed_results['Predicted_Anomaly'], target_names=['Normal', 'Anomaly']))

AttributeError: 'int' object has no attribute 'total_seconds'